# Topic 20f - Transforms, window functions, and binning

Practice for sections 7 and 8 of the [main lesson](topic20_data_wrangling.ipynb).

Three ways to derive new columns:

| Need | Tool |
|---|---|
| A value computed from the row's **group** | `groupby().transform()` |
| A value computed from **neighbouring rows** | `cumsum`, `shift`, `pct_change`, `rolling` |
| A **category** made from a number | `cut`, `qcut`, `np.where`, `map` |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_dir():
    """Locate the lesson's data folder from any sensible working directory."""
    for candidate in [Path.cwd() / "data",
                      Path.cwd() / "data_wrangling" / "data",
                      Path.cwd().parent / "data"]:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find orders.csv - check your data folder.")


DATA_DIR = find_data_dir()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["figure.dpi"] = 110

MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def load_analysis_table():
    """The cleaned, validated, joined table built in the main lesson."""
    orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])
    customers = pd.read_csv(DATA_DIR / "customers.csv")
    products = pd.read_csv(DATA_DIR / "products.csv")
    regions = pd.read_csv(DATA_DIR / "regions.csv")

    customers = customers.copy()
    customers["region"] = (customers["region"].str.strip()
                                              .str.replace(r"\s+", " ", regex=True)
                                              .str.title())
    customers = customers.merge(regions, on="region", how="left", validate="m:1")
    products = products.drop_duplicates(subset="product_id", keep="first")

    table = (orders
             .merge(customers, on="customer_id", how="left", validate="m:1")
             .merge(products, on="product_id", how="left", validate="m:1"))
    table["revenue_ghs"] = (table["quantity"] * table["unit_price_ghs"]).round(2)
    table["cost_ghs"] = (table["quantity"] * table["unit_cost_ghs"]).round(2)
    table["profit_ghs"] = (table["revenue_ghs"] - table["cost_ghs"]).round(2)
    table["month"] = table["order_date"].dt.strftime("%b")
    return table


analysis = load_analysis_table()
print(f"analysis table: {analysis.shape[0]} rows x {analysis.shape[1]} columns")

## 1. `agg` collapses, `transform` broadcasts

The difference is the shape of what comes back.

In [ ]:
work = analysis[["order_id", "region", "channel", "revenue_ghs"]].copy()

collapsed = work.groupby("region")["revenue_ghs"].sum()
broadcast = work.groupby("region")["revenue_ghs"].transform("sum")

print(f"groupby().sum()       -> {len(collapsed):>5} values (one per region)")
print(f"groupby().transform() -> {len(broadcast):>5} values (one per order)")
print("\ntransform repeats the group's answer on every row of that group,")
print("which is exactly what you need to add it as a column.")

In [ ]:
work["region_total"] = work.groupby("region")["revenue_ghs"].transform("sum")
work["pct_of_region"] = (work["revenue_ghs"] / work["region_total"] * 100).round(2)
work["region_mean"] = work.groupby("region")["revenue_ghs"].transform("mean").round(2)
work["above_region_mean"] = work["revenue_ghs"] > work["region_mean"]

print(work.sort_values("pct_of_region", ascending=False).head(5).to_string(index=False))
print(f"\nOrders above their own region's mean: {work['above_region_mean'].sum()}"
      f" of {len(work)}")

## 2. Ranking within groups

`rank()` needs `method=` to say how ties are handled, and a nullable `Int64` if any group is missing.

In [ ]:
work["rank_in_region"] = (work.groupby("region")["revenue_ghs"]
                              .rank(ascending=False, method="min")
                              .astype("Int64"))

print("Top 2 orders in three regions:")
top = work[work["rank_in_region"] <= 2].sort_values(["region", "rank_in_region"])
print(top[top["region"].isin(["Ashanti", "Greater Accra", "Volta"])]
      [["order_id", "region", "revenue_ghs", "rank_in_region"]].to_string(index=False))

print(f"\nOrders with no rank: {work['rank_in_region'].isna().sum()}")
print("Those are the orphan orders - no known customer, so no region to rank within.")

## 3. Window functions

These read neighbouring rows, so the table **must be sorted first**. `shift(1)` returns whatever row is above; on unsorted data that is not "last month".

In [ ]:
monthly = (analysis.groupby(analysis["order_date"].dt.to_period("M"))
                   .agg(revenue_ghs=("revenue_ghs", "sum"),
                        orders=("order_id", "count"))
                   .reset_index())
monthly["order_date"] = monthly["order_date"].astype(str)
monthly = monthly.sort_values("order_date").reset_index(drop=True)   # SORT FIRST

monthly["running_total"] = monthly["revenue_ghs"].cumsum().round(2)
monthly["previous"] = monthly["revenue_ghs"].shift(1).round(2)
monthly["change_pct"] = (monthly["revenue_ghs"].pct_change() * 100).round(1)
monthly["rolling_3m"] = monthly["revenue_ghs"].rolling(3).mean().round(2)

print(monthly.to_string(index=False))

The `NaN` values in the first rows are correct, not a defect: there is no month before January, and a three-month average needs three months. Filling them with zero would invent a month in which nothing was sold and drag every later average down.

## 4. Binning: `cut` versus `qcut`

They answer different questions.

- `cut` uses **boundaries you choose**: "an order over GHS 500 is large." Fixed, meaningful, group sizes vary.
- `qcut` uses **quantiles**: "the largest quarter of orders." Relative, group sizes equal, boundaries move with the data.

In [ ]:
banded = analysis[["order_id", "region", "revenue_ghs"]].copy()

banded["size_band"] = pd.cut(banded["revenue_ghs"],
                             bins=[0, 50, 200, 500, np.inf],
                             labels=["Micro", "Small", "Medium", "Large"],
                             right=False)
banded["quartile"] = pd.qcut(banded["revenue_ghs"], q=4,
                             labels=["Q1", "Q2", "Q3", "Q4"])
banded["is_large"] = np.where(banded["revenue_ghs"] >= 500, "Yes", "No")

print("cut() - fixed boundaries, varying group sizes:")
print(banded["size_band"].value_counts().sort_index().to_string())
print("\nqcut() - equal group sizes, varying boundaries:")
print(banded["quartile"].value_counts().sort_index().to_string())
print("\nBoundaries qcut chose:")
print(pd.qcut(banded["revenue_ghs"], q=4).cat.categories)

### `right=False` and the boundary problem

`pd.cut` includes the right edge by default, so `bins=[0, 50, 200]` creates `(0, 50]` and `(50, 200]` - a value of exactly 50 lands in the **first** band. With `right=False` you get `[0, 50)` and `[50, 200)`, and 50 lands in the second.

Neither is wrong. But you must know which you chose, because it decides where round numbers go - and round numbers are common in prices.

In [ ]:
edges = pd.DataFrame({"amount": [49.99, 50.00, 50.01]})
edges["right_true"] = pd.cut(edges["amount"], bins=[0, 50, 200],
                             labels=["low", "high"])
edges["right_false"] = pd.cut(edges["amount"], bins=[0, 50, 200],
                              labels=["low", "high"], right=False)
print(edges.to_string(index=False))
print("\nExactly 50.00 changes band depending on the setting.")

## 5. `map()` for recoding labels

In [ ]:
channel_groups = {"Shop": "In person", "Field Agent": "In person",
                  "Phone": "Remote", "Online": "Remote"}
banded_channels = analysis[["order_id", "channel"]].copy()
banded_channels["contact"] = banded_channels["channel"].map(channel_groups)

print(banded_channels["contact"].value_counts().to_string())
print("\nA value missing from the dictionary becomes NaN, so always check:")
print("unmapped rows:", banded_channels["contact"].isna().sum())

## 6. Practice

1. Add a column giving each order's **share of its category's** total revenue.
2. Add a column ranking products by total revenue within their category.
3. Compute each region's **cumulative** revenue through 2023, sorted by date, and plot one region's running total as a line chart.
4. Split customers into three tiers by total revenue using `qcut` (`Bronze`, `Silver`, `Gold`). How much revenue does each tier account for?
5. Answer in markdown: why do the three tiers hold roughly equal **numbers** of customers but very different **amounts** of revenue?

In [ ]:
# YOUR CODE HERE

**Your answer to question 5:**

*(Write your answer here.)*

---
## Where this fits

- [Back to the main lesson](topic20_data_wrangling.ipynb)
- Previous: [20e - Reshaping with melt and pivot](topic20e_reshaping_melt_and_pivot.ipynb)
- Next: [20g - Wrangling practice](topic20g_wrangling_practice.ipynb)